<a href="https://colab.research.google.com/github/xabreo/Beatrice-V2-Trainer-Colab/blob/main/Beatrice_V2_Simple_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beatrice V2 Simple Trainer

In [ ]:
#@title Beatrice V2 Simple Trainer
from pathlib import Path
from IPython.display import HTML, display
import sys

display(HTML((Path.cwd() / "beatrice_v2_simple_trainer.html").read_text(encoding="utf-8")))

sys.path.insert(0, str(Path.cwd()))
from beatrice_trainer import tab1_environment, tab2_project, tab3_initialize, tab4_settings, tab5_training, _bridge_json

try:
    from google.colab import output as colab_output
    def beatrice_ui_action(action, payload=None):
        payload = payload or {}
        try:
            if action == "setup_environment": tab1_environment(); return _bridge_json("Environment setup complete.")
            if action == "restart_runtime": return _bridge_json("Restart the Colab runtime, then rerun this cell.")
            if action == "verify_environment": tab1_environment(); return _bridge_json("Environment verification complete.")
            if action == "mount_drive": tab1_environment(); return _bridge_json("Google Drive mounted.")
            if action == "verify_drive": tab1_environment(); return _bridge_json("Drive verification complete.")
            if action in {"scan_project", "find_zip", "scan_checkpoints"}: tab2_project(); return _bridge_json("Project scan complete.")
            if action in {"save_normal_settings", "load_config", "validate_config"}: tab4_settings(); return _bridge_json("Configuration saved.")
            if action == "reset_config": tab3_initialize(); return _bridge_json("Configuration reset to defaults.")
            if action == "preflight":
                from beatrice_trainer import CONFIG_PATH, STATE_FILE
                if not CONFIG_PATH.is_file(): return _bridge_json("Configuration missing. Run Tab 3.")
                if not STATE_FILE.is_file(): return _bridge_json("Project state missing. Run Tab 2.")
                return _bridge_json("Pre-flight ready.")
            if action == "import_trainer":
                from beatrice_trainer import REPO
                return _bridge_json("Trainer already available.") if REPO.is_dir() else _bridge_json("Run Tab 5 to clone trainer.")
            if action == "prepare_workspace":
                from beatrice_trainer import LOCAL_OUTPUT; LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
                return _bridge_json(f"Workspace ready: {LOCAL_OUTPUT}")
            if action == "start_training": tab5_training(); return _bridge_json("Training complete.")
            if action == "stop_training": return _bridge_json("Stop requested.")
            if action in {"start_watchdog", "stop_watchdog", "sync_checkpoints", "final_backup"}: return _bridge_json("Owned by Tab 5.")
            if action == "gpu_status":
                import subprocess
                r = subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw", "--format=csv,noheader,nounits"], capture_output=True, text=True)
                if r.returncode != 0: return _bridge_json("nvidia-smi failed.")
                p = [x.strip() for x in r.stdout.strip().split(",")]
                if len(p) >= 6: return {"gpu":p[0],"utilization":p[1]+"%","memory":p[2]+" / "+p[3]+" MiB","temperature":p[4]+" C","power":p[5]+" W","message":"GPU refreshed."}
                return _bridge_json(r.stdout.strip())
            if action == "training_status": return {"status":"Idle","message":"OK"}
            return _bridge_json(f"Unknown: {action}")
        except Exception as e: return _bridge_json(f"Error: {e}")
    colab_output.register_callback("beatrice_ui_action", beatrice_ui_action)
    print("Backend connected.")
except Exception: print("Colab bridge unavailable.")